In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import boto3
import io
import pyodbc
import urllib
import sys

In [ ]:
print(pyodbc.drivers())

### Configuration

In [ ]:
SOURCE_DB = "<database_name>" # i.e AdventureWorks
SOURCE_SCHEMA = "<schema_name>" # i.e Sales
TABLES = ['Customer', 'SalesPerson', 'CreditCard', 'SalesOrderDetail', 'Currency'] # replace table list with yours

#DB Credentials
USER = "<username>" # i.e sa
PASS = "<password>" 
SERVER = "<server_name" # i.e localhost

# The exact driver name you found above or the one installed locally for you
DRIVER = "ODBC Driver 18 for SQL Server"

# CONSTRUCT CONNECTION STRING 
# We use urllib.parse.quote_plus to handle the spaces in the driver name safely
params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={SOURCE_DB};"
    f"UID={USER};"
    f"PWD={PASS};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)
SQL_CONN_STR = f"mssql+pyodbc:///?odbc_connect={params}"

# Initialize engines
sql_engine = create_engine(SQL_CONN_STR)

s3_client = boto3.client('s3')

S3_BUCKET = "ck-datatech-landing"

### Validate Connection

In [ ]:
def test_sql_connection():
    print(f"Testing connection to {SOURCE_DB} using {DRIVER}...")
    try:
        with sql_engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("Connection successful!\n")
        return True
    except Exception as e:
        print(f"Connection failed: {e}")
        return False

In [ ]:
test_sql_connection()

### Migrate data

In [ ]:
def migrate_in_memory():
    if not test_sql_connection():
        sys.exit(1)

    for table in TABLES:
        full_table_path = f"[{SOURCE_DB}].[{SOURCE_SCHEMA}].[{table}]"
        print(f"Migrating {full_table_path} -> S3...")

        query = f"SELECT * FROM {full_table_path}"
        chunk_size = 100000

        try:
            for i, chunk in enumerate(pd.read_sql_query(query, sql_engine, chunksize=chunk_size)):

                # Convert ALL datetime columns to microseconds
                for col in chunk.select_dtypes(include=['datetime64']).columns:
                    chunk[col] = chunk[col].dt.floor('us')

                # Buffer and upload once per chunk, after all columns are processed
                buffer = io.BytesIO()
                
                chunk.to_parquet(
                    buffer,
                    engine='pyarrow',
                    compression='snappy',
                    index=False
                )
                buffer.seek(0)

                s3_key = f"sqlserver/{SOURCE_SCHEMA}/{table}/part_{i}.parquet"
                s3_client.upload_fileobj(buffer, S3_BUCKET, s3_key)

                print(f"Uploaded part {i} to s3://{S3_BUCKET}/{s3_key}")

        except Exception as e:
            print(f"Error migrating {table}: {e}")

if __name__ == "__main__":
    migrate_in_memory()